# 20. Nested Subgraph Multi-Agent Architecture
**Industry:** Insurance

Build a LangGraph system where a top-level supervisor graph calls a nested subgraph (its own internal multi-step workflow) as a single node.

In [1]:
!pip install langgraph langchain langchain-openai

  Using cached langchain_core-1.5.3-py3-none-any.whl.metadata (4.7 kB)
INFO: pip is looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain-1.3.14-py3-none-any.whl.metadata (6.1 kB)
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain_openai-1.4.1-py3-none-any.whl.metadata (3.4 kB)
Using cached langchain-1.3.14-py3-none-any.whl (139 kB)
Using cached langchain_openai-1.4.1-py3-none-any.whl (122 kB)
Using cached langchain_core-1.5.3-py3-none-any.whl (561 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.86
    Uninstalling langchain-core-0.3.86:
      Successfully uninstalled langchain-core-0.3.86
  Attempting uninstall: langchain-openai
    Found existing installation: langchain-openai 0.3.35
    Uninstalling langchain-

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
langchain-experimental 0.4.2 requires langchain-community<1.0.0,>=0.4.2, but you have langchain-community 0.3.31 which is incompatible.
langchain-google-firestore 0.5.0 requires langchain-core<1.0.0,>=0.1.1, but you have langchain-core 1.5.3 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class ClaimState(TypedDict):
    claim_id: str
    status: str
    damage_cost: int

# --- Inner Graph: Damage Assessment ---
def research_photos(state: ClaimState):
    return {"status": "photos_researched"}

def estimate_cost(state: ClaimState):
    return {"damage_cost": 500}

inner_graph = StateGraph(ClaimState)
inner_graph.add_node("research", research_photos)
inner_graph.add_node("estimate", estimate_cost)
inner_graph.add_edge(START, "research")
inner_graph.add_edge("research", "estimate")
inner_graph.add_edge("estimate", END)
inner_app = inner_graph.compile()

# --- Outer Graph: Claims Supervisor ---
def supervisor_node(state: ClaimState):
    return {"status": "assessing_damage"}

outer_graph = StateGraph(ClaimState)
outer_graph.add_node("supervisor", supervisor_node)
outer_graph.add_node("damage_assessment", inner_app)
outer_graph.add_edge(START, "supervisor")
outer_graph.add_edge("supervisor", "damage_assessment")
outer_graph.add_edge("damage_assessment", END)
outer_app = outer_graph.compile()

for event in outer_app.stream({"claim_id": "C123", "status": "new", "damage_cost": 0}):
    print(event)

{'supervisor': {'status': 'assessing_damage'}}
{'damage_assessment': {'claim_id': 'C123', 'status': 'photos_researched', 'damage_cost': 500}}
